# Semantic Segmentation
Today we will cover:
- Theory: FCN, U-Net
- Exercise: Segment medical images
- Mini Project: Tumor segmentation prototype

## Theory: Semantic Segmentation Architectures

### Fully Convolutional Networks (FCN)
- Replace fully connected layers with convolutional layers.
- Output is a dense pixel-wise prediction map.
- Key Idea: End-to-end segmentation using convolutional layers.

### U-Net (2015)
- Encoder-decoder architecture with skip connections.
- Encoder: Downsampling path to capture context.
- Decoder: Upsampling path to recover spatial resolution.
- Widely used in medical image segmentation.
- Key Idea: Combine low-level and high-level features for precise segmentation.

## Setup: Medical Image Dataset Loader

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

# Example dataset class (replace with actual medical dataset path)
class MedicalDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        return image, mask

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

train_dataset = MedicalDataset('./medical/train/images', './medical/train/masks', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print("Loaded medical dataset with", len(train_dataset), "images")

## Exercise: U-Net for Medical Image Segmentation

In [ ]:
import torch.nn as nn

# Define a simple U-Net architecture
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

model = UNet()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop (simplified)
for epoch in range(1, 3):
    running_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {running_loss/len(train_loader):.4f}")

## Mini Project: Tumor Segmentation Prototype

In [ ]:
import matplotlib.pyplot as plt

# Evaluate on a batch
images, masks = next(iter(train_loader))
outputs = model(images.to(device))
preds = torch.sigmoid(outputs).cpu().detach()

# Display original image, ground truth mask, and predicted mask
fig, axs = plt.subplots(1, 3, figsize=(12,4))
axs[0].imshow(images[0].permute(1,2,0))
axs[0].set_title('Original Image')
axs[1].imshow(masks[0][0], cmap='gray')
axs[1].set_title('Ground Truth Mask')
axs[2].imshow(preds[0][0], cmap='gray')
axs[2].set_title('Predicted Mask')
plt.show()